In [ ]:
# ============================================================
# EXPERIMENT 8A: SUPPORT VECTOR MACHINE (SVM) FROM SCRATCH
# Synthetic Student Performance Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report

# Step 1 & 2: Create Synthetic Dataset and Display Info
df = pd.DataFrame({
    "Study_Hours": [1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 2, 3, 4, 6, 7],
    "Attendance": [50, 55, 60, 58, 65, 70, 72, 75, 78, 80, 85, 88, 90, 92, 95, 52, 62, 68, 82, 87],
    "Class": [-1, -1, -1, -1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1]
})
print(f"Student Performance Dataset:\n{df}")
print(f"\nDataset Shape:\n{df.shape}\n\nDataset Description:\n{df.describe()}\n\nClass Distribution:\n{df['Class'].value_counts()}")

# Step 3, 4 & 5: Separate Features, Train-Test Split, and Feature Scaling
X, y = df[["Study_Hours", "Attendance"]].values, df["Class"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled, X_test_scaled = scaler.fit_transform(X_train), scaler.transform(X_test)

# Step 6: Define SVM Class from Scratch
class SVMFromScratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iterations=1000):
        self.lr, self.lam, self.n_iter = learning_rate, lambda_param, n_iterations
        self.weights = self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights, self.bias = np.zeros(n_features), 0
        for _ in range(self.n_iter):
            total_loss = 0
            for i, x_i in enumerate(X):
                cond = y[i] * (np.dot(x_i, self.weights) + self.bias) >= 1
                if cond:
                    dw, db, hl = 2 * self.lam * self.weights, 0, 0
                else:
                    dw = 2 * self.lam * self.weights - y[i] * x_i
                    db, hl = -y[i], 1 - y[i] * (np.dot(x_i, self.weights) + self.bias)
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                total_loss += max(0, hl)
            self.loss_history.append(total_loss / n_samples + self.lam * np.dot(self.weights, self.weights))

    def predict(self, X):
        return np.where(np.dot(X, self.weights) + self.bias >= 0, 1, -1)

# Step 7 & 8: Create and Train SVM Model
svm = SVMFromScratch(learning_rate=0.01, lambda_param=0.01, n_iterations=1000)
svm.fit(X_train_scaled, y_train)
print("\nSVM Training Completed Successfully.")

# Step 9 & 10: Display Parameters and Make Predictions
print(f"\n===================================\nSVM MODEL PARAMETERS\n===================================\nWeights: {svm.weights}\nBias: {svm.bias}")

y_pred = svm.predict(X_test_scaled)
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
print(f"\n===================================\nACTUAL VS PREDICTED\n===================================\n{results}")

# Steps 11-16: Calculate and Display Performance Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

print(f"\n===================================\nSVM CLASSIFICATION PERFORMANCE\n===================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}\nPrecision: {round(prec, 4)}\nRecall: {round(rec, 4)}\nF1 Score: {round(f1, 4)}")

# Steps 17 & 18: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
print(f"\n===================================\nCONFUSION MATRIX\n===================================\n{cm}")
print(f"\n===================================\nCLASSIFICATION REPORT\n===================================\n{classification_report(y_test, y_pred, labels=[-1, 1], target_names=['Fail', 'Pass'], zero_division=0)}")

# Step 19: Plot Training Loss
plt.figure(figsize=(8, 5))
plt.plot(svm.loss_history), plt.xlabel("Iteration"), plt.ylabel("Loss")
plt.title("SVM Training Loss"), plt.grid(True), plt.tight_layout(), plt.show()

# Step 20: Plot Decision Boundary
X_all_scaled = scaler.transform(X)
plt.figure(figsize=(8, 6))
plt.scatter(X_all_scaled[y == -1, 0], X_all_scaled[y == -1, 1], marker="o", label="Fail (-1)")
plt.scatter(X_all_scaled[y == 1, 0], X_all_scaled[y == 1, 1], marker="x", label="Pass (+1)")

x_vals = np.linspace(X_all_scaled[:, 0].min() - 1, X_all_scaled[:, 0].max() + 1, 100)
y_vals = -((svm.weights[0] * x_vals + svm.bias) / svm.weights[1])
plt.plot(x_vals, y_vals, label="Decision Boundary")
plt.plot(x_vals, (1 - svm.weights[0] * x_vals - svm.bias) / svm.weights[1], linestyle="--", label="Positive Margin")
plt.plot(x_vals, (-1 - svm.weights[0] * x_vals - svm.bias) / svm.weights[1], linestyle="--", label="Negative Margin")

plt.xlabel("Scaled Study Hours"), plt.ylabel("Scaled Attendance"), plt.title("SVM Decision Boundary")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 21: Predict New Student
new_student = np.array([[5, 78]])
new_pred = svm.predict(scaler.transform(new_student))[0]
print(f"\n===================================\nNEW STUDENT PREDICTION\n===================================")
print(f"Study Hours: {new_student[0][0]}\nAttendance: {new_student[0][1]}\nPredicted Class: {'Pass' if new_pred == 1 else 'Fail'}")

In [ ]:
# ============================================================
# EXPERIMENT 8B: SUPPORT VECTOR MACHINE USING SCIKIT-LEARN
# Breast Cancer Dataset (RBF Kernel)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names, target_names = cancer.feature_names, cancer.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
print(f"Dataset Shape:\n{df.shape}\n\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{feature_names}\n\nTarget Names:\n{target_names}\n\nClass Distribution:\n{df['Target'].value_counts()}\n\nDataset Description:\n{df.describe()}")

# Step 4-6: Separate Features, Train-Test Split, and Feature Scaling
X, y = df[feature_names].values, df["Target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled, X_test_scaled = scaler.fit_transform(X_train), scaler.transform(X_test)

# Step 7-8: Create and Train SVM Model (RBF Kernel)
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42).fit(X_train_scaled, y_train)
print("\nSVM Model Training Completed Successfully.")

# Step 9: Display Support Vector Information
print(f"\n===================================\nSUPPORT VECTOR INFORMATION\n===================================")
print(f"Number of Support Vectors: {svm_model.n_support_}\nTotal Support Vectors: {len(svm_model.support_vectors_)}")

# Step 10-13: Predictions, Probabilities, and Actual vs Predicted
y_pred = svm_model.predict(X_test_scaled)
y_prob = svm_model.predict_proba(X_test_scaled)
dec_vals = svm_model.decision_function(X_test_scaled)

results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred, "Decision_Value": dec_vals, "Probability_Benign": y_prob[:, 1]})
print(f"\n===================================\nACTUAL VS PREDICTED\n===================================\n{results.head(20)}")

# Steps 14-18: Calculate and Display Performance Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"\n===================================\nSVM CLASSIFICATION PERFORMANCE\n===================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}\nPrecision: {round(prec, 4)}\nRecall: {round(rec, 4)}\nF1 Score: {round(f1, 4)}")

# Steps 19-20: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\n===================================\nCONFUSION MATRIX\n===================================\n{cm}")
print(f"\n===================================\nCLASSIFICATION REPORT\n===================================\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Step 21: Plot Confusion Matrix
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("SVM Confusion Matrix"), plt.colorbar()
ticks = np.arange(len(target_names))
plt.xticks(ticks, target_names), plt.yticks(ticks, target_names)
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout(), plt.show()

# Step 22: Actual vs Predicted Plot
n_samples = min(30, len(y_test))
idx = np.arange(n_samples)
plt.figure(figsize=(10, 5))
plt.scatter(idx, y_test[:n_samples], marker="o", label="Actual Class")
plt.scatter(idx, y_pred[:n_samples], marker="x", label="Predicted Class")
plt.xlabel("Test Sample Index"), plt.ylabel("Class"), plt.title("SVM: Actual vs Predicted Classes")
plt.yticks([0, 1], target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 23: Predict a New Sample
new_sample = scaler.transform(X_test[0].reshape(1, -1))
new_pred, new_prob = svm_model.predict(new_sample), svm_model.predict_proba(new_sample)
print(f"\n===================================\nNEW SAMPLE PREDICTION\n===================================")
print(f"Predicted Class: {target_names[new_pred[0]]}\n\nPrediction Probabilities:")
for i in range(len(target_names)):
    print(f"{target_names[i]}: {new_prob[0][i]:.4f}")